# Soccer Lineup Problem in Pure Python
## Five Exact Methods for Book Problem 2.13


## Problem Statement

In book problem `2.13`, the coach of Atlético Andes must assign `10` outfield players to three
positions: defense, midfield and forward. The model maximizes the total shooting score subject to:

- at least `4` defenders, `3` midfielders and `3` forwards,
- average ball skill and average speed of at least `2`,
- mutual-exclusion and implication constraints among a few players,
- forbidden `(player, position)` pairs.

The notebooks below solve the exact model printed in the book.


In [ ]:
from __future__ import annotations

from functools import lru_cache, wraps
from itertools import combinations
from time import perf_counter


def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = perf_counter() - start
        return result, elapsed

    return wrapper


PLAYERS = list(range(1, 13))
POSITIONS = {1: "defense", 2: "midfield", 3: "forward"}
HB = {1: 1, 2: 3, 3: 3, 4: 4, 5: 2, 6: 4, 7: 3, 8: 4, 9: 2, 10: 2, 11: 3, 12: 2}
SHOT = {1: 2, 2: 3, 3: 3, 4: 4, 5: 1, 6: 2, 7: 4, 8: 2, 9: 3, 10: 3, 11: 4, 12: 1}
SPEED = {1: 4, 2: 2, 3: 4, 4: 3, 5: 2, 6: 2, 7: 2, 8: 3, 9: 4, 10: 4, 11: 3, 12: 2}
FORBIDDEN = {(1, 3), (2, 1), (4, 1), (5, 3), (6, 3), (7, 1), (7, 2), (9, 1), (11, 1), (11, 3), (12, 1)}
KEY_PLAYERS = [1, 2, 4, 5, 6, 9, 11]
KEY_BITS = {player: index for index, player in enumerate(KEY_PLAYERS)}
EXPECTED_OBJECTIVE = 28


def assignment_vector(solution):
    vector = []
    defenders = set(solution["defenders"])
    midfielders = set(solution["midfielders"])
    forwards = set(solution["forwards"])
    for player in PLAYERS:
        if player in defenders:
            vector.append(1)
        elif player in midfielders:
            vector.append(2)
        elif player in forwards:
            vector.append(3)
        else:
            vector.append(0)
    return tuple(vector)


def choose_better(left, right):
    if left is None:
        return right
    if right is None:
        return left
    left_key = (left["objective"], assignment_vector(left))
    right_key = (right["objective"], assignment_vector(right))
    return right if right_key > left_key else left


def selection_logic(selected):
    return (
        int(5 in selected) == 1 - int(11 in selected)
        and int(1 in selected) == int(6 in selected)
        and int(1 in selected) == int(9 in selected)
        and int(2 in selected) == 1 - int(4 in selected)
    )


def make_solution(defenders, midfielders, forwards):
    defenders = tuple(sorted(defenders))
    midfielders = tuple(sorted(midfielders))
    forwards = tuple(sorted(forwards))
    selected = set(defenders) | set(midfielders) | set(forwards)
    if len(defenders) != 4 or len(midfielders) != 3 or len(forwards) != 3:
        return None
    if len(selected) != 10:
        return None
    if any((player, 1) in FORBIDDEN for player in defenders):
        return None
    if any((player, 2) in FORBIDDEN for player in midfielders):
        return None
    if any((player, 3) in FORBIDDEN for player in forwards):
        return None
    if sum(HB[player] for player in selected) < 20:
        return None
    if sum(SPEED[player] for player in selected) < 20:
        return None
    if not selection_logic(selected):
        return None
    return {
        "defenders": defenders,
        "midfielders": midfielders,
        "forwards": forwards,
        "bench": tuple(sorted(set(PLAYERS) - selected)),
        "objective": sum(SHOT[player] for player in selected),
    }


def solution_from_assignment_tuple(codes):
    defenders = [player for player, code in zip(PLAYERS, codes) if code == 1]
    midfielders = [player for player, code in zip(PLAYERS, codes) if code == 2]
    forwards = [player for player, code in zip(PLAYERS, codes) if code == 3]
    return make_solution(defenders, midfielders, forwards)


@timed
def solve_soccer_naive():
    best = None
    for defenders in combinations(PLAYERS, 4):
        if any((player, 1) in FORBIDDEN for player in defenders):
            continue
        remaining_after_defense = [player for player in PLAYERS if player not in defenders]
        for midfielders in combinations(remaining_after_defense, 3):
            if any((player, 2) in FORBIDDEN for player in midfielders):
                continue
            remaining_after_midfield = [player for player in remaining_after_defense if player not in midfielders]
            for forwards in combinations(remaining_after_midfield, 3):
                candidate = make_solution(defenders, midfielders, forwards)
                best = choose_better(best, candidate)
    return best


@timed
def solve_soccer_backtracking():
    best = None

    def upper_bound(index, current_score, selected_count):
        need = 10 - selected_count
        remaining_scores = sorted((SHOT[player] for player in PLAYERS[index:]), reverse=True)
        return current_score + sum(remaining_scores[:need])

    def backtrack(index, assignments, defenders, midfielders, forwards, hb_sum, speed_sum, current_score):
        nonlocal best
        selected_count = defenders + midfielders + forwards
        remaining_players = len(PLAYERS) - index
        if defenders > 4 or midfielders > 3 or forwards > 3:
            return
        if (4 - defenders) + (3 - midfielders) + (3 - forwards) > remaining_players:
            return
        if selected_count > 10 or 10 - selected_count > remaining_players:
            return
        if best is not None and upper_bound(index, current_score, selected_count) < best["objective"]:
            return
        if index == len(PLAYERS):
            candidate = solution_from_assignment_tuple(tuple(assignments))
            best = choose_better(best, candidate)
            return

        player = PLAYERS[index]
        options = [0]
        if (player, 1) not in FORBIDDEN:
            options.append(1)
        if (player, 2) not in FORBIDDEN:
            options.append(2)
        if (player, 3) not in FORBIDDEN:
            options.append(3)

        for code in options:
            assignments.append(code)
            backtrack(
                index + 1,
                assignments,
                defenders + int(code == 1),
                midfielders + int(code == 2),
                forwards + int(code == 3),
                hb_sum + (HB[player] if code else 0),
                speed_sum + (SPEED[player] if code else 0),
                current_score + (SHOT[player] if code else 0),
            )
            assignments.pop()

    backtrack(0, [], 0, 0, 0, 0, 0, 0)
    return best


def find_feasible_assignment_for_selected(selected_players):
    best = None
    for defenders in combinations(selected_players, 4):
        if any((player, 1) in FORBIDDEN for player in defenders):
            continue
        remaining_after_defense = [player for player in selected_players if player not in defenders]
        for midfielders in combinations(remaining_after_defense, 3):
            if any((player, 2) in FORBIDDEN for player in midfielders):
                continue
            forwards = [player for player in remaining_after_defense if player not in midfielders]
            candidate = make_solution(defenders, midfielders, forwards)
            best = choose_better(best, candidate)
    return best


@timed
def solve_soccer_reduced_search():
    best = None
    for bench_pair in [(2, 5), (2, 11), (4, 5), (4, 11)]:
        selected = [player for player in PLAYERS if player not in bench_pair]
        candidate = find_feasible_assignment_for_selected(selected)
        best = choose_better(best, candidate)
    return best


@timed
def solve_soccer_dynamic_programming():
    @lru_cache(maxsize=None)
    def dp(index, defenders, midfielders, forwards, hb_sum, speed_sum, selection_mask):
        selected_count = defenders + midfielders + forwards
        remaining_players = len(PLAYERS) - index
        if defenders > 4 or midfielders > 3 or forwards > 3:
            return None
        if (4 - defenders) + (3 - midfielders) + (3 - forwards) > remaining_players:
            return None
        if selected_count > 10 or 10 - selected_count > remaining_players:
            return None
        if index == len(PLAYERS):
            selected = {player for player in KEY_PLAYERS if (selection_mask >> KEY_BITS[player]) & 1}
            if (defenders, midfielders, forwards) != (4, 3, 3):
                return None
            if hb_sum < 20 or speed_sum < 20:
                return None
            if not selection_logic(selected):
                return None
            return (0, ())

        player = PLAYERS[index]
        best_local = None
        options = [0]
        if (player, 1) not in FORBIDDEN:
            options.append(1)
        if (player, 2) not in FORBIDDEN:
            options.append(2)
        if (player, 3) not in FORBIDDEN:
            options.append(3)
        for code in options:
            next_mask = selection_mask
            if player in KEY_BITS and code != 0:
                next_mask |= 1 << KEY_BITS[player]
            tail = dp(
                index + 1,
                defenders + int(code == 1),
                midfielders + int(code == 2),
                forwards + int(code == 3),
                min(20, hb_sum + (HB[player] if code else 0)),
                min(20, speed_sum + (SPEED[player] if code else 0)),
                next_mask,
            )
            if tail is None:
                continue
            candidate = (tail[0] + (SHOT[player] if code else 0), (code,) + tail[1])
            if best_local is None or candidate > best_local:
                best_local = candidate
        return best_local

    packed = dp(0, 0, 0, 0, 0, 0, 0)
    return solution_from_assignment_tuple(packed[1])


@timed
def solve_soccer_branch_and_bound():
    best = solve_soccer_reduced_search()[0]

    def upper_bound(index, current_score, selected_count):
        need = 10 - selected_count
        remaining_scores = sorted((SHOT[player] for player in PLAYERS[index:]), reverse=True)
        return current_score + sum(remaining_scores[:need])

    def branch(index, assignments, defenders, midfielders, forwards, current_score):
        nonlocal best
        selected_count = defenders + midfielders + forwards
        remaining_players = len(PLAYERS) - index
        if defenders > 4 or midfielders > 3 or forwards > 3:
            return
        if (4 - defenders) + (3 - midfielders) + (3 - forwards) > remaining_players:
            return
        if selected_count > 10 or 10 - selected_count > remaining_players:
            return
        if upper_bound(index, current_score, selected_count) < best["objective"]:
            return
        if index == len(PLAYERS):
            candidate = solution_from_assignment_tuple(tuple(assignments))
            best = choose_better(best, candidate)
            return

        player = PLAYERS[index]
        options = [0]
        if (player, 1) not in FORBIDDEN:
            options.append(1)
        if (player, 2) not in FORBIDDEN:
            options.append(2)
        if (player, 3) not in FORBIDDEN:
            options.append(3)
        options.sort(key=lambda code: (SHOT[player] if code else 0, code), reverse=True)

        for code in options:
            assignments.append(code)
            branch(
                index + 1,
                assignments,
                defenders + int(code == 1),
                midfielders + int(code == 2),
                forwards + int(code == 3),
                current_score + (SHOT[player] if code else 0),
            )
            assignments.pop()

    branch(0, [], 0, 0, 0, 0)
    return best


## 1. Naive Enumeration


In [ ]:
naive_result, naive_time = solve_soccer_naive()
print("Naive result:", naive_result)
print(f"Elapsed time: {naive_time:.6f} seconds")
assert naive_result["objective"] == EXPECTED_OBJECTIVE


## 2. Backtracking With Pruning


In [ ]:
backtracking_result, backtracking_time = solve_soccer_backtracking()
print("Backtracking result:", backtracking_result)
print(f"Elapsed time: {backtracking_time:.6f} seconds")
assert backtracking_result["objective"] == EXPECTED_OBJECTIVE


## 3. Constraint-Driven Reduced Search


In [ ]:
print("Exact reduction: the bench must contain one player from {2, 4} and one from {5, 11}.")
reduced_result, reduced_time = solve_soccer_reduced_search()
print("Reduced-search result:", reduced_result)
print(f"Elapsed time: {reduced_time:.6f} seconds")
assert reduced_result["objective"] == EXPECTED_OBJECTIVE


## 4. Dynamic Programming


In [ ]:
dp_result, dp_time = solve_soccer_dynamic_programming()
print("Dynamic-programming result:", dp_result)
print(f"Elapsed time: {dp_time:.6f} seconds")
assert dp_result["objective"] == EXPECTED_OBJECTIVE


## 5. Branch And Bound


In [ ]:
bnb_result, bnb_time = solve_soccer_branch_and_bound()
print("Branch-and-bound result:", bnb_result)
print(f"Elapsed time: {bnb_time:.6f} seconds")
assert bnb_result["objective"] == EXPECTED_OBJECTIVE


## Summary


In [ ]:
rows = [
    ("Naive enumeration", naive_result, naive_time),
    ("Backtracking with pruning", backtracking_result, backtracking_time),
    ("Constraint-driven reduced search", reduced_result, reduced_time),
    ("Dynamic programming", dp_result, dp_time),
    ("Branch and Bound", bnb_result, bnb_time),
]

for method_name, result, elapsed in rows:
    print(
        f"{method_name:35s} objective={result['objective']:2d} "
        f"bench={result['bench']} defenders={result['defenders']} "
        f"midfield={result['midfielders']} forwards={result['forwards']} time={elapsed:.6f}s"
    )

assert all(result["objective"] == EXPECTED_OBJECTIVE for _, result, _ in rows)
print("\nAll five exact methods reach an optimal shooting score of 28.")
